<a href="https://colab.research.google.com/github/Puja528/Uji-Validitas-dan-Realibilitas/blob/main/Uji_Validitas_dan_Reliabitas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pingouin factor_analyzer -q

In [4]:
!pip install --upgrade numpy scipy pingouin factor_analyzer

In [5]:
import pandas as pd, numpy as np, pingouin as pg
from scipy import stats
from factor_analyzer import FactorAnalyzer, calculate_kmo, calculate_bartlett_sphericity

/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [8]:
from google.colab import files
up = files.upload()
nama_file = list(up.keys())[0]
print("File terbaca:", nama_file)

Saving Data_TIB.xlsx to Data_TIB (1).xlsx
File terbaca: Data_TIB (1).xlsx


In [9]:
JUMLAH_KOLOM_DEMOGRAFI = 6  # karena indeks 0-5 adalah data demografi/umum

df = pd.read_excel(nama_file)

# Menghapus kolom indeks ke-20 (pertanyaan durasi jam) sebelum memisahkan butir
df_bersih = df.drop(columns=[df.columns[20]])

butir = df_bersih.iloc[:, JUMLAH_KOLOM_DEMOGRAFI:]
print("Jumlah responden:", len(df_bersih))
print("Jumlah butir    :", butir.shape[1])
print("Data kosong     :", int(butir.isna().sum().sum()))

Jumlah responden: 41
Jumlah butir    : 29
Data kosong     : 0


In [10]:
skala = {
    "Sangat Tidak Setuju (STS)": 1,
    "Tidak Setuju (TS)": 2,
    "Netral (N)": 3,
    "Setuju (S)": 4,
    "Sangat Setuju (SS)": 5,
}
num = butir.replace(skala).apply(pd.to_numeric, errors="coerce")

# Harus 0. Jika lebih dari 0, ada jawaban yang tidak terbaca.
print("Sel yang gagal dikonversi:", int(num.isna().sum().sum()))
num.head()

Sel yang gagal dikonversi: 0


,Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan platform AI hampir setiap kali mendapatkan tugas akademik.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari solusi atas permasalahan atau soal perkuliahan.],Penggunaan AI pada tugas atau projek kuliah [Saya menggunakan AI untuk mencari ide atau inspirasi dalam mengerjakan tugas./projek],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kesulitan memulai mengerjakan tugas tanpa bantuan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya merasa kurang yakin dengan hasil tugas apabila tidak menggunakan AI.],Penggunaan AI pada tugas atau projek kuliah [Saya percaya bahwa AI dapat memberikan informasi yang membantu penyelesaian tugas.],Penggunaan AI pada tugas atau projek kuliah [Saya memeriksa kembali kebenaran informasi yang diberikan AI sebelum menggunakannya.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya merasa perlu menggunakan AI setiap kali mendapatkan tugas atau proyek kuliah.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [aya merasa kesulitan menyelesaikan tugas kuliah apabila tidak menggunakan bantuan AI.],Aspek Psikologis penggunaan AI pada tugas atau projek kuliah [Saya cenderung mengikuti saran atau jawaban AI tanpa banyak mempertimbangkan alternatif lainnya (Non-AI).],...,Kebiasaan penggunaan teknologi digital [Saya jarang memeriksa kembali hasil dari AI karena sudah percaya terhadap ketepatan jawabannya.],Prokrastinasi akademik. [Saya cenderung menunggu hingga mendekati deadline untuk mulai mengerjakan tugas.],Prokrastinasi akademik. [Saya sering menunda pengerjaan tugas karena merasa dapat dikerjakan dengan mudah dan cepat dengan menggunakan AI],Prokrastinasi akademik. [Saya sering menyelesaikan tugas kuliah dalam waktu yang sangat dekat dengan batas pengumpulan.],Prokrastinasi akademik. [Saya sering mengerjakan beberapa tugas sekaligus karena sebelumnya menunda pengerjaannya.],Rsspon emosional [Saya merasa tertekan ketika harus menyelesaikan tugas dalam waktu singkat akibat menunda pengerjaannya.],Rsspon emosional [Saya merasa panik ketika beberapa tugas yang saya tunda memiliki batas waktu pengumpulan yang berdekatan.],Rsspon emosional [Saya merasa kurang puas terhadap hasil tugas yang dikerjakan karena terburu-buru mendekati deadline.],Rsspon emosional [Saya merasa lega setelah berhasil mengumpulkan tugas meskipun pengerjaannya dilakukan AI saat mendekati deadline.],Rsspon emosional [Saya merasa menyesal karena hasil tugas mungkin bisa lebih baik apabila dikerjakan lebih awal.]
0,4,2,5,2,4,2,5,4,2,2,...,1,1,2,1,2,4,2,5,4,5
1,4,5,5,4,4,4,5,4,4,2,...,1,2,1,1,1,4,5,4,2,4
2,2,2,4,1,1,4,5,2,1,1,...,1,2,2,2,3,2,2,4,2,5
3,4,5,5,4,2,4,5,2,4,1,...,2,1,5,1,5,1,2,4,2,4
4,5,5,5,5,2,5,5,5,5,2,...,1,5,5,5,5,1,1,5,5,4


In [12]:
import re

dimensi = {}
for kolom in butir.columns:
    nama = re.match(r"^(.*?)\[", kolom).group(1).strip()
    dimensi.setdefault(nama, []).append(kolom)

for nama, daftar in dimensi.items():
    print(f"{nama}: {len(daftar)} butir")

Penggunaan AI pada tugas atau projek kuliah: 7 butir
Aspek Psikologis penggunaan AI pada tugas atau projek kuliah: 7 butir
Kebiasaan penggunaan teknologi digital: 6 butir
Prokrastinasi akademik.: 4 butir
Rsspon emosional: 5 butir


In [13]:
n = len(num)
t = stats.t.ppf(0.975, n - 2)
r_tabel = t / np.sqrt((n - 2) + t**2)
print(f"n = {n} | r tabel (α=0,05; df={n-2}) = {r_tabel:.3f}")

hasil_validitas = []
for nama, daftar in dimensi.items():
  x = num[daftar]
  total = x.sum(axis=1)

  for i, kolom in enumerate(daftar, 1):
    r, p = stats.pearsonr(x[kolom], total)
    r_koreksi, _ = stats.pearsonr(x[kolom], total - x[kolom])
    hasil_validitas.append({
        "Dimensi": nama,
        "Butir": f"B{i}",
        "r hitung": round(r, 3),
        "Corrected r": round(r_koreksi, 3),
        "p-value": round(p, 4),
        "Keterangan": "Valid" if r > r_tabel else "Tidak valid",
})

n = 41 | r tabel (α=0,05; df=39) = 0.308


In [14]:
tabel_validitas = pd.DataFrame(hasil_validitas)
tabel_validitas

,Dimensi,Butir,r hitung,Corrected r,p-value,Keterangan
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.710,0.557,0.0000,Valid
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.605,0.440,0.0000,Valid
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.394,0.137,0.0109,Valid
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.788,0.588,0.0000,Valid
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.638,0.339,0.0000,Valid
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.422,0.236,0.0060,Valid
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.086,-0.062,0.5936,Tidak valid
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.655,0.437,0.0000,Valid
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.698,0.497,0.0000,Valid
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.581,0.390,0.0001,Valid


In [17]:
def cronbach_alpha(x):
  k = x.shape[1]
  return k / (k - 1) * (1 - x.var(ddof=1).sum() / x.sum(axis=1).var(ddof=1))

def kategori(a):
  return ("Sangat baik" if a >= 0.90 else "Baik" if a >= 0.80
          else "Reliabel" if a >= 0.70 else "Tidak reliabel")

hasil_reliabilitas = []
alpha_dihapus = []
for nama, daftar in dimensi.items():
  x = num[daftar]
  a = cronbach_alpha(x)
  hasil_reliabilitas.append({"Dimensi": nama, "Jumlah butir": len(daftar),
                             "Cronbach's alpha": round(a, 3), "Kategori": kategori(a)})
  for i, kolom in enumerate(daftar, 1):
    if len(daftar) > 2:
      alpha_dihapus.append({"Dimensi": nama, "Butir": f"B{i}",
                            "Alpha jika dihapus": round(cronbach_alpha(x.drop(columns=kolom)), 3)})

tabel_reliabilitas = pd.DataFrame(hasil_reliabilitas)
tabel_alpha_dihapus = pd.DataFrame(alpha_dihapus)
display(tabel_reliabilitas)
display(tabel_alpha_dihapus)

,Dimensi,Jumlah butir,Cronbach's alpha,Kategori
0,Penggunaan AI pada tugas atau projek kuliah,7,0.604,Tidak reliabel
1,Aspek Psikologis penggunaan AI pada tugas atau...,7,0.635,Tidak reliabel
2,Kebiasaan penggunaan teknologi digital,6,0.534,Tidak reliabel
3,Prokrastinasi akademik.,4,0.738,Reliabel
4,Rsspon emosional,5,0.559,Tidak reliabel


,Dimensi,Butir,Alpha jika dihapus
0,Penggunaan AI pada tugas atau projek kuliah,B1,0.490
1,Penggunaan AI pada tugas atau projek kuliah,B2,0.534
2,Penggunaan AI pada tugas atau projek kuliah,B3,0.627
3,Penggunaan AI pada tugas atau projek kuliah,B4,0.441
4,Penggunaan AI pada tugas atau projek kuliah,B5,0.569
5,Penggunaan AI pada tugas atau projek kuliah,B6,0.591
6,Penggunaan AI pada tugas atau projek kuliah,B7,0.645
7,Aspek Psikologis penggunaan AI pada tugas atau...,B1,0.565
8,Aspek Psikologis penggunaan AI pada tugas atau...,B2,0.542
9,Aspek Psikologis penggunaan AI pada tugas atau...,B3,0.585


In [16]:
gabung = tabel_validitas.merge(tabel_alpha_dihapus, on=["Dimensi", "Butir"], how="left")

with pd.ExcelWriter("hasil_uji_validitas_reliabilitas.xlsx") as w:
  gabung.to_excel(w, sheet_name="Validitas", index=False)
  tabel_reliabilitas.to_excel(w, sheet_name="Reliabilitas", index=False)

files.download("hasil_uji_validitas_reliabilitas.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>